# Climate-GEA gained-peaks / gained-genes — ALL bioclim axes (bio1..bio19)

Per (climate axis, model) row: WZA-corrected clq0.9-block significant peaks by class
(SNP / non-SNP / SV) and the gain over SNPs (`n_nonsnp_not_snp`, `n_sv_not_snp` + pct),
at Bonferroni and BH-FDR. Then the union list of genes under non-SNP-only / SV-only peaks
across all axes, annotated with Ensembl-Plants function. LFMM is the calibrated model;
binomial/Kendall are inflated references (read as upper bounds).

In [1]:

import os, sys
import numpy as np, pandas as pd
os.chdir("/global/scratch/users/tbellg/kmate"); sys.path.insert(0, "analysis/grenenet_gea")
sys.path.insert(0, "analysis/grenenet_gea/gea_newpanel")
import lib, blocks_clq09
GNP = "analysis/grenenet_gea/gea_newpanel/results"; MA = f"{GNP}/multiaxis_fresh"
MODELS = ["binomial", "kendall", "lfmm"]; CLASSES = ["snp", "nonsnp", "sv"]
AXES = [f"bio{i}" for i in range(1, 20)]

def wza_path(model, cls, axis):
    return f"{GNP}/wza_{model}_{cls}_clq09.csv" if axis == "bio1" \
        else f"{MA}/wza_{model}_{cls}_{axis}_clq09.csv"

def sig_sets(model, cls, axis, q=0.05):
    f = wza_path(model, cls, axis)
    if not os.path.exists(f):
        return None
    w = pd.read_csv(f).rename(columns={"index": "block"}); w["block"] = w.block.astype(str)
    w = w[w.Z_pVal.notna()]
    if not len(w):
        return set(), set()
    p = w.Z_pVal.to_numpy()
    return set(w.loc[p < 0.05/len(w), "block"]), set(w.loc[lib.bh(p) < q, "block"])

# cache sig sets per (axis,model,class,threshold)
S = {}
avail = []
for ax in AXES:
    ok = all(sig_sets(m, c, ax) is not None for m in MODELS for c in CLASSES)
    if not ok:
        continue
    avail.append(ax)
    for m in MODELS:
        for c in CLASSES:
            b, f = sig_sets(m, c, ax); S[(ax, m, c, 0)] = b; S[(ax, m, c, 1)] = f
print(f"axes available: {len(avail)}/{len(AXES)} -> {avail}")


axes available: 19/19 -> ['bio1', 'bio2', 'bio3', 'bio4', 'bio5', 'bio6', 'bio7', 'bio8', 'bio9', 'bio10', 'bio11', 'bio12', 'bio13', 'bio14', 'bio15', 'bio16', 'bio17', 'bio18', 'bio19']


## Table 1 — Bonferroni, per (axis, model)

In [2]:

def gain_rows(thr):
    rows = []
    for ax in avail:
        for m in MODELS:
            s = {c: S[(ax, m, c, thr)] for c in CLASSES}
            row = {"axis": ax, "model": m, "n_snp": len(s["snp"]), "n_nonsnp": len(s["nonsnp"]), "n_sv": len(s["sv"])}
            for b in ("nonsnp", "sv"):
                new = s[b] - s["snp"]; row[f"n_{b}_not_snp"] = len(new)
                row[f"pct_{b}_not_snp"] = round(len(new)/len(s[b]), 3) if s[b] else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

gain_bonf = gain_rows(0); gain_fdr = gain_rows(1)
gain_bonf.to_csv(f"{GNP}/snp_vs_nonsnp/gea_classgain_multiaxis_bonf.csv", index=False)
gain_fdr.to_csv(f"{GNP}/snp_vs_nonsnp/gea_classgain_multiaxis_fdr.csv", index=False)
print("per-(axis,model) tables:", gain_bonf.shape)

gain_bonf

per-(axis,model) tables: (57, 9)


,axis,model,n_snp,n_nonsnp,n_sv,n_nonsnp_not_snp,pct_nonsnp_not_snp,n_sv_not_snp,pct_sv_not_snp
0,bio1,binomial,3,5,0,4,0.800,0,NaN
1,bio1,kendall,3,4,0,2,0.500,0,NaN
2,bio1,lfmm,13,11,4,5,0.455,4,1.0
3,bio2,binomial,1,0,3,0,NaN,3,1.0
4,bio2,kendall,15,12,0,6,0.500,0,NaN
5,bio2,lfmm,17,7,1,4,0.571,0,0.0
6,bio3,binomial,5,3,6,2,0.667,6,1.0
7,bio3,kendall,7,8,0,5,0.625,0,NaN
8,bio3,lfmm,9,5,0,3,0.600,0,NaN
9,bio4,binomial,0,0,0,0,NaN,0,NaN


## Table 2 — BH-FDR, per (axis, model)

In [3]:
gain_fdr

,axis,model,n_snp,n_nonsnp,n_sv,n_nonsnp_not_snp,pct_nonsnp_not_snp,n_sv_not_snp,pct_sv_not_snp
0,bio1,binomial,11,34,2,27,0.794,2,1.000
1,bio1,kendall,19,19,0,14,0.737,0,NaN
2,bio1,lfmm,66,52,5,33,0.635,5,1.000
3,bio2,binomial,1,7,3,7,1.000,3,1.000
4,bio2,kendall,97,99,0,52,0.525,0,NaN
5,bio2,lfmm,74,49,1,29,0.592,0,0.000
6,bio3,binomial,23,13,9,7,0.538,9,1.000
7,bio3,kendall,94,72,0,37,0.514,0,NaN
8,bio3,lfmm,54,35,0,17,0.486,0,NaN
9,bio4,binomial,0,0,0,0,NaN,0,NaN


## Gained genes across all axes (Ensembl-annotated)

In [4]:

bmap = {r.block: (r.chrom, int(r.start_pos), int(r.end_pos)) for r in blocks_clq09.load_blocks().itertuples()}
genes = lib.load_genes(); FLANK = 2000

def gained_genes(thr):
    hits = {}
    for cls in ("nonsnp", "sv"):
        blk2 = {}
        for ax in avail:
            for m in MODELS:
                for b in (S[(ax, m, cls, thr)] - S[(ax, m, "snp", thr)]):
                    d = blk2.setdefault(b, {"axes": set(), "models": set()})
                    d["axes"].add(ax); d["models"].add(m)
        for b, info in blk2.items():
            if b not in bmap:
                continue
            c, s0, e0 = bmap[b]
            inb = genes[(genes.chrom == c) & (genes.start <= e0) & (genes.end >= s0)]
            flk = genes[(genes.chrom == c) & (genes.start <= e0+FLANK) & (genes.end >= s0-FLANK)]
            flk = flk[~flk.gene.isin(inb.gene)]
            for gid, otype in [(g, "in_block") for g in inb.gene] + [(g, "flank2kb") for g in flk.gene]:
                d = hits.setdefault((gid, cls), dict(gene=gid, klass=cls, chrom=c, overlap=otype,
                                                     blocks=set(), axes=set(), models=set()))
                d["blocks"].add(b); d["axes"].update(info["axes"]); d["models"].update(info["models"])
                if otype == "in_block":
                    d["overlap"] = "in_block"
    rows = [dict(gene=d["gene"], klass=d["klass"], chrom=d["chrom"], overlap=d["overlap"],
                 n_blocks=len(d["blocks"]), n_axes=len(d["axes"]), n_models=len(d["models"]),
                 axes=";".join(sorted(d["axes"])), models=";".join(sorted(d["models"]))) for d in hits.values()]
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["klass", "n_axes", "n_models", "gene"], ascending=[True, False, False, True]).reset_index(drop=True)
    return df

genes_bonf = gained_genes(0); genes_fdr = gained_genes(1)
gb = f"{GNP}/snp_vs_nonsnp/gea_gained_genes_multiaxis_bonf.csv"
gf = f"{GNP}/snp_vs_nonsnp/gea_gained_genes_multiaxis_fdr.csv"
genes_bonf.to_csv(gb, index=False); genes_fdr.to_csv(gf, index=False)
print(f"gained genes (union all axes) — Bonf {len(genes_bonf)}, FDR {len(genes_fdr)}")
# Ensembl-Plants function annotation (adds symbol + function columns in place)
import subprocess
subprocess.run([sys.executable, "analysis/grenenet_gea/gea_newpanel/annotate_gained_genes.py", gb, gf], check=False)
genes_bonf = pd.read_csv(gb).fillna(""); genes_fdr = pd.read_csv(gf).fillna("")


gained genes (union all axes) — Bonf 333, FDR 1169


fetching 1016 genes from Ensembl…
annotated analysis/grenenet_gea/gea_newpanel/results/snp_vs_nonsnp/gea_gained_genes_multiaxis_bonf.csv: 333 rows, 326 with function
annotated analysis/grenenet_gea/gea_newpanel/results/snp_vs_nonsnp/gea_gained_genes_multiaxis_fdr.csv: 1169 rows, 1148 with function


### Genes — Bonferroni (non-SNP-only / SV-only, any axis)

In [5]:
genes_bonf

,gene,klass,chrom,overlap,n_blocks,n_axes,n_models,axes,models,symbol,function
0,AT3G06340,nonsnp,Chr3,in_block,1,11,2,bio1;bio10;bio11;bio12;bio14;bio17;bio4;bio5;b...,binomial;kendall,AT3G06340,DNAJ heat shock N-terminal domain-containing p...
1,AT3G06350,nonsnp,Chr3,flank2kb,1,11,2,bio1;bio10;bio11;bio12;bio14;bio17;bio4;bio5;b...,binomial;kendall,MEE32,"dehydroquinate dehydratase, putative / shikima..."
2,AT1G72510,nonsnp,Chr1,flank2kb,1,8,2,bio1;bio10;bio12;bio14;bio18;bio19;bio7;bio9,binomial;kendall,AT1G72510,DUF1677 family protein (DUF1677)
3,AT1G72520,nonsnp,Chr1,flank2kb,1,8,2,bio1;bio10;bio12;bio14;bio18;bio19;bio7;bio9,binomial;kendall,LOX4,PLAT/LH2 domain-containing lipoxygenase family...
4,AT5G24110,nonsnp,Chr5,flank2kb,1,7,3,bio1;bio10;bio14;bio15;bio17;bio18;bio6,binomial;kendall;lfmm,WRKY30,WRKY DNA-binding protein 30
...,...,...,...,...,...,...,...,...,...,...,...
328,AT5G28210,sv,Chr5,flank2kb,1,1,1,bio1,lfmm,AT5G28210,mRNA capping enzyme family protein
329,AT5G28220,sv,Chr5,in_block,1,1,1,bio1,lfmm,AT5G28220,Protein prenylyltransferase superfamily protein
330,AT5G28235,sv,Chr5,in_block,1,1,1,bio1,lfmm,AT5G28235,Ulp1 protease family protein
331,AT5G28237,sv,Chr5,in_block,1,1,1,bio1,lfmm,AT5G28237,Pyridoxal-5'-phosphate-dependent enzyme family...


### Genes — FDR (non-SNP-only / SV-only, any axis)

In [6]:
genes_fdr

,gene,klass,chrom,overlap,n_blocks,n_axes,n_models,axes,models,symbol,function
0,AT5G24130,nonsnp,Chr5,in_block,3,14,3,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18...,binomial;kendall;lfmm,AT5G24130,polypyrimidine tract-binding-like protein
1,AT4G21220,nonsnp,Chr4,flank2kb,1,14,2,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18...,binomial;kendall,LpxD2,Trimeric LpxA-like enzymes superfamily protein
2,AT4G21230,nonsnp,Chr4,in_block,1,14,2,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18...,binomial;kendall,CRK27,cysteine-rich RLK (RECEPTOR-like protein kinas...
3,AT5G24110,nonsnp,Chr5,flank2kb,1,13,3,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18...,binomial;kendall;lfmm,WRKY30,WRKY DNA-binding protein 30
4,AT5G24120,nonsnp,Chr5,in_block,1,13,3,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18...,binomial;kendall;lfmm,SIGE,sigma factor E
...,...,...,...,...,...,...,...,...,...,...,...
1164,AT5G38090,sv,Chr5,flank2kb,1,1,1,bio1,lfmm,AT5G38090,uncharacterized protein
1165,AT5G38100,sv,Chr5,flank2kb,1,1,1,bio1,lfmm,AT5G38100,S-adenosyl-L-methionine-dependent methyltransf...
1166,AT5G51890,sv,Chr5,flank2kb,1,1,1,bio6,kendall,AT5G51890,Peroxidase superfamily protein
1167,AT5G51900,sv,Chr5,in_block,1,1,1,bio6,kendall,AT5G51900,Cytochrome P450 family protein
